# Fine-tune E5-base — 6 epoch (thử nghiệm)

Fine-tune **`intfloat/multilingual-e5-base`** với **6 epoch** để xem metric có tiếp tục tăng sau 2–3 epoch hay đã overfit rõ.

| | 1 ep | 2 ep | 3 ep | Notebook này (6 ep) |
|---|---|---|---|---|
| Model | `e5_base_finetuned_final/` | `e5_base_finetuned_2ep_final/` | `e5_base_finetuned_3ep_final/` | `e5_base_finetuned_6ep_final/` |
| Metrics | `metrics_e5_base.json` | `metrics_e5_base_2epochs.json` | `metrics_e5_base_3epochs.json` | `metrics_e5_base_6epochs.json` |

**Cần GPU Colab** (T4 trở lên; A100/L4 nếu train lâu). Thời gian train ~6× so với 1 epoch (~1–2 giờ tùy GPU).

> **`load_best_model_at_end=True`**: model lưu vào `e5_base_finetuned_6ep_final/` là checkpoint có **valid loss thấp nhất** trong 6 epoch — không nhất thiết epoch 6.
>
> Theo kết quả trước (valid loss epoch 2 < epoch 1), khả năng overfit tăng dần từ epoch 3+. Cell loss giúp xác nhận.

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo và kiểm tra GPU

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

GITHUB_REPO_URL = "https://github.com/YOUR_USER/llm_provider_benchmarking.git"  # ← sửa URL
REPO_DIR = "/content/llm_provider_benchmarking"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, REPO_DIR], check=True)

SCRIPTS_DIR = f"{REPO_DIR}/embedding_project/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

os.chdir(REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Cấu hình — 6 epoch

In [ ]:
from pathlib import Path
from model_presets import get_preset

PRESET = get_preset("e5-base")
PROJECT_ROOT = Path(REPO_DIR) / "embedding_project"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_GPU = torch.cuda.is_available()
EPOCHS = 6
BATCH_SIZE = 8 if USE_GPU else 2
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
LEARNING_RATE = PRESET.learning_rate
WARMUP_RATIO = PRESET.warmup_ratio

FINAL_DIR = MODELS_DIR / "e5_base_finetuned_6ep_final"
CHECKPOINT_DIR = MODELS_DIR / "e5-base-6ep"
METRICS_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_6epochs.json"
METRICS_1EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base.json"
METRICS_2EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_2epochs.json"
METRICS_3EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_3epochs.json"

print("Base model:", PRESET.base_model)
print("Final dir:", FINAL_DIR)
print(f"epochs={EPOCHS} | batch={BATCH_SIZE} | fp16={FP16} | max_seq={MAX_SEQ_LENGTH} | lr={LEARNING_RATE}")

## 4) Load train / valid

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / "train_cleaned.jsonl")
valid_rows = load_jsonl(DATA_DIR / "valid_cleaned.jsonl")
print("train:", len(train_rows), "| valid:", len(valid_rows))

train_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in train_rows])
valid_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in valid_rows])

## 5) Fine-tune 6 epoch

Loss: `MultipleNegativesRankingLoss`. Checkpoint mỗi epoch; giữ tối đa 6 checkpoint gần nhất.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

model = SentenceTransformer(PRESET.base_model)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=20,
    save_total_limit=6,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    run_name="e5-base-vi-6epochs-colab",
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

train_result = trainer.train()
model.save(str(FINAL_DIR))
print("Saved model (best valid loss):", FINAL_DIR)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Train summary:", train_result)

## 6) Loss theo epoch + biểu đồ

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_by_epoch = {}
valid_by_epoch = {}

for entry in log_history:
    ep = entry.get("epoch")
    if ep is None:
        continue
    if "loss" in entry:
        train_by_epoch[ep] = entry["loss"]
    if "eval_loss" in entry:
        valid_by_epoch[ep] = entry["eval_loss"]

epochs = sorted(set(train_by_epoch) | set(valid_by_epoch))
rows = []
for ep in epochs:
    rows.append({
        "epoch": int(ep) if float(ep).is_integer() else ep,
        "train_loss": train_by_epoch.get(ep),
        "valid_loss": valid_by_epoch.get(ep),
    })

df_log = pd.DataFrame(rows)
display(df_log)

if not df_log.empty and df_log["valid_loss"].notna().any():
    best_idx = df_log["valid_loss"].idxmin()
    best_row = df_log.loc[best_idx]
    print(f"Valid loss thấp nhất: epoch {best_row['epoch']} = {best_row['valid_loss']:.6f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_log["epoch"], df_log["train_loss"], marker="o", label="train_loss")
    ax.plot(df_log["epoch"], df_log["valid_loss"], marker="s", label="valid_loss")
    ax.axvline(best_row["epoch"], color="green", linestyle="--", alpha=0.6, label="best valid")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("E5-base — 6 epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7) Đánh giá trên test (fine-tuned 6 epoch — best checkpoint)

In [ ]:
!python embedding_project/scripts/evaluate_embedding_model.py \
  --preset e5-base \
  --only-finetuned \
  --finetuned-model embedding_project/models/e5_base_finetuned_6ep_final \
  --output embedding_project/outputs/evaluation/metrics_e5_base_6epochs.json \
  --no-cache

## 8) So sánh 1 / 2 / 3 / 6 epoch

In [ ]:
import json

def load_metrics(path):
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

paths = {
    "1 epoch": METRICS_1EP_FILE,
    "2 epoch": METRICS_2EP_FILE,
    "3 epoch": METRICS_3EP_FILE,
    "6 epoch": METRICS_FILE,
}
loaded = {label: load_metrics(p) for label, p in paths.items()}
ft = {label: (m or {}).get("finetuned", {}) for label, m in loaded.items()}

metrics = ["Precision@10", "Recall@10", "MRR@10", "NDCG@10"]
rows = []
for k in metrics:
    row = {"Metric": k}
    for label in paths:
        row[label] = ft[label].get(k)
    v2, v6 = ft["2 epoch"].get(k), ft["6 epoch"].get(k)
    if v2 is not None and v6 is not None and v2 != 0:
        row["Δ (6 vs 2)"] = f"{(v6 - v2) / v2 * 100:+.1f}%"
    else:
        row["Δ (6 vs 2)"] = "—"
    rows.append(row)

df_cmp = pd.DataFrame(rows)
display(df_cmp)

missing = [label for label, m in loaded.items() if not (m or {}).get("finetuned")]
if missing:
    print("Thiếu metrics:", ", ".join(missing))
    print("Upload JSON vào embedding_project/outputs/evaluation/ trên Colab.")

if ft["2 epoch"] and ft["6 epoch"]:
    better = sum(1 for k in metrics if ft["6 epoch"].get(k, 0) > ft["2 epoch"].get(k, 0))
    worse = sum(1 for k in metrics if ft["6 epoch"].get(k, 0) < ft["2 epoch"].get(k, 0))
    print(f"6 epoch vs 2 epoch: tốt hơn {better}/{len(metrics)}, kém hơn {worse}/{len(metrics)}.")
    if worse >= 3:
        print("→ Gợi ý: giữ bản 2 epoch cho production.")
    elif better >= 3:
        print("→ Gợi ý: thử index Qdrant với model 6 epoch.")

## 9) Tải model về máy (zip)

In [ ]:
import shutil
from google.colab import files

assert FINAL_DIR.is_dir(), f"Chưa có model: {FINAL_DIR}"

zip_path = shutil.make_archive("/content/e5_base_finetuned_6ep_final", "zip", root_dir=str(FINAL_DIR))
print("Zip:", zip_path)
files.download(zip_path)

## 10) (Tùy chọn) Tải metrics JSON

In [ ]:
from google.colab import files

if METRICS_FILE.is_file():
    files.download(str(METRICS_FILE))
else:
    print("Chưa có file metrics — chạy cell đánh giá trước.")

## 11) (Tùy chọn) Index Qdrant sau khi tải model về máy

```bash
python vector_db/03_index_to_qdrant.py --recreate \
  --collection products_vi_e5_6ep \
  --model-path embedding_project/models/e5_base_finetuned_6ep_final \
  --e5-prefix --encode-batch-size 8

python vector_db/04_search_test.py --collection products_vi_e5_6ep \
  --model-path embedding_project/models/e5_base_finetuned_6ep_final \
  --e5-prefix --query "giày chạy bộ nam" --top-k 5
```